# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding #4 — "The Freshness Multiplier" (365+ day content refreshed shows 3.2x health boost, 57x impressions)

Where the label comes from: the "refreshed" vs. "not refreshed" split appears to be an observed editorial action — someone decided which old pages got updated — not a randomized assignment. The comparison (10.7 → 34.5 health, 71 → 4,039 impressions) is a before/after on pages that were chosen for refresh, not a random sample of old pages.

Methodology question: does the validation design carry the causal-sounding language in the writeup ("refresh timing is one of the strongest measured levers")? If editors preferentially refreshed pages that already showed signs of recoverable demand (backlinks still active, a seasonal topic about to trend again, internal links still healthy), the 57x jump could be selection bias dressed up as a refresh effect — the paper would be measuring "which pages editors correctly bet on," not "what refreshing does to a random old page." The paper's own limitations section flags this as observational, which is the right caveat, but the finding box's framing ("one of the strongest measured levers available") reads more confidently than an unrandomized before/after comparison can really support. A stronger design would compare refreshed pages against a matched control group of similarly-old, similarly-visible pages that weren't touched, rather than only showing the before/after of the treated group.

ML Appendix — "What Predicts Health?" (Random Forest: Average Position 43%, Impressions 32%, Scroll Depth 15%)

Where the label comes from: Health Score is explicitly defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts) — a formula, not an observed outcome. Three of the model's top features (position, impressions, and implicitly CTR) are literal components of the target.

Methodology question: does the validation design carry the claim that these are "predictors"? A held-out test split protects against overfitting noise, but it can't protect against a feature that's structurally identical to part of the label — a train/test split doesn't catch that kind of leakage the way it catches memorization, because the leaked signal generalizes perfectly to the test set too. The paper does the responsible thing by calling this "descriptive rather than causal" in the caption, but I'd push the caveat one step further: this isn't just "not causal," it's closer to "not really prediction" in the ML sense at all — it's closer to solving for the known weights of a formula. The honest reframe would be "the model recovers the known composition of Health Score," not "Average Position predicts Health Score." This is exactly the leakage pattern from our Week 3 exercise (a feature derived from the target), just at the level of an entire scoring rubric instead of one column.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# --- FIX: Define 'feat' DataFrame here. Replace with your actual data loading. ---
# Placeholder: Replace with your actual data loading, e.g., pd.read_csv('your_data.csv')
# For now, creating a sample DataFrame with required columns to avoid KeyError
np.random.seed(42)
num_rows = 100
feat = pd.DataFrame({
    "avg_impressions_28d": np.random.randint(100, 1000, num_rows),
    "avg_position_28d": np.random.uniform(1.0, 20.0, num_rows),
    "days_since_update": np.random.randint(0, 365, num_rows),
    "avg_clicks_28d": np.random.randint(10, 200, num_rows),
    "client_key": np.random.randint(1, 10, num_rows) # Example client keys for grouping
})

target_col = "avg_clicks_28d"
features = ["avg_impressions_28d", "avg_position_28d", "days_since_update"]

# --- BEFORE: naive random split (what a first pass often looks like) ---
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    feat[features], feat[target_col], test_size=0.2, random_state=42
)
rf_naive = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(X_train_naive, y_train_naive)
naive_r2 = r2_score(y_test_naive, rf_naive.predict(X_test_naive))

# --- AFTER: grouped-by-client split (from w05) ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_key"]))
train_df, test_df = feat.iloc[train_idx], feat.iloc[test_idx]
rf_grouped = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(
    train_df[features], train_df[target_col]
)
grouped_r2 = r2_score(test_df[target_col], rf_grouped.predict(test_df[features]))

comparison = pd.DataFrame({
    "split_type": ["Naive random split", "Grouped-by-client split"],
    "R2": [naive_r2, grouped_r2]
})
print(comparison)
print(f"\nGap: {naive_r2 - grouped_r2:+.3f} — a positive gap here means the naive split was "
      f"overstating performance by letting the same client's pages leak across train/test.")

                split_type        R2
0       Naive random split -0.098946
1  Grouped-by-client split -0.397500

Gap: +0.299 — a positive gap here means the naive split was overstating performance by letting the same client's pages leak across train/test.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

def quick_score(df, feature_cols, target_col):
    data = df.dropna(subset=feature_cols + [target_col])
    X, y = data[feature_cols], data[target_col]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    return r2_score(yte, model.predict(Xte))

final_features = ["avg_impressions_28d", "avg_position_28d", "days_since_update"]
baseline = quick_score(feat, final_features, target_col)
print(f"Baseline R2 with final feature set: {baseline:.3f}")

for f in final_features:
    remaining = [c for c in final_features if c != f]
    score = quick_score(feat, remaining, target_col)
    print(f"Without '{f}': R2 = {score:.3f} (delta: {score - baseline:+.3f})")

# Explicit check: confirm ctr_28d (flagged leaky in w03) is NOT in final_features
assert "ctr_28d" not in final_features, "Leaky feature ctr_28d still present!"
print("\nConfirmed: ctr_28d excluded from final model inputs.")

Baseline R2 with final feature set: -0.044
Without 'avg_impressions_28d': R2 = -0.038 (delta: +0.005)
Without 'avg_position_28d': R2 = -0.054 (delta: -0.010)
Without 'days_since_update': R2 = 0.008 (delta: +0.052)

Confirmed: ctr_28d excluded from final model inputs.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

"This model predicts which pages will drive more traffic after a refresh."
Rewritten: "On this mid-panel month's anonymized sample, the model's ranking of pages by predicted engagement showed a directional, measured improvement over the Week-4 rule-based baseline (R² of X vs Y under a grouped split) — this is decision-support for prioritizing a review queue, not a causal prediction of what a refresh will do to any individual page's future traffic."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.